In [16]:
import os, re, json, time, pickle
import numpy as np
import pandas as pd
import faiss
from dotenv import load_dotenv
from google import genai
from google.genai import types
from sentence_transformers import SentenceTransformer

GEN_MODEL = "gemini-3.8-flash"
K = 5
DATA = "../data"

with open(f"{DATA}/all_chunks_v2.pkl", "rb") as f:
    all_chunks = pickle.load(f)
index = faiss.read_index(f"{DATA}/all_index_v2.faiss")
assert index.ntotal == len(all_chunks), "index and chunks are out of sync"

embedder = SentenceTransformer("all-MiniLM-L6-v2")

load_dotenv("../.env")
llm = genai.Client(api_key=os.getenv("GEMINI_API_KEY"),
                   http_options=types.HttpOptions(timeout=120_000))

def search(query, k=K):
    q = embedder.encode([query], normalize_embeddings=True)
    scores, idxs = index.search(np.array(q, dtype="float32"), k)
    return [{"score": float(s), "text": all_chunks[i]["text"], "source": all_chunks[i]["source"]}
            for s, i in zip(scores[0], idxs[0])]

def ask(query, k=K, max_retries=3):
    results = search(query, k)
    context = "\n\n".join(f"[{i+1}] {r['text']}" for i, r in enumerate(results))
    prompt = f"""You are a research assistant. Answer using the context below.
If the context doesn't contain the answer, say "I don't have enough information."
Cite sources as [1], [2], etc.

Context:
{context}

Question: {query}

Answer:"""
    for attempt in range(max_retries):
        try:
            resp = llm.models.generate_content(model=GEN_MODEL, contents=prompt)
            return (resp.text or "").strip(), results
        except Exception as e:
            msg = str(e)
            if "429" in msg or "RESOURCE_EXHAUSTED" in msg:
                raise
            if attempt < max_retries - 1 and any(s in msg for s in
                    ("503", "504", "UNAVAILABLE", "DEADLINE_EXCEEDED", "disconnected", "timed out")):
                wait = 5 * (attempt + 1)
                print(f"   transient error, retrying in {wait}s...")
                time.sleep(wait)
                continue
            raise

print("Chunks:", len(all_chunks), "| Index:", index.ntotal)

Loading weights: 100%|█████████████████████| 103/103 [00:00<00:00, 2598.42it/s]


Chunks: 726 | Index: 726


In [17]:

eval_set = [
    {"q": "What is autoregressive retrieval augmentation?",        "expected": ["2506.06962v3.pdf"]},
    {"q": "How does MusT-RAG handle music question answering?",    "expected": ["2507.23334v2.pdf"]},
    {"q": "What datasets were used to evaluate the AR-RAG paper?", "expected": ["2506.06962v3.pdf"]},
        {"q": "What is the main limitation of standard RAG systems?",  "expected": ["2510.25621v1.pdf", "2601.07528v2.pdf"]},
    {"q": "How does agentic RAG improve Quran question answering?","expected": ["2601.07528v2.pdf"]},
       {"q": "What is the role of FAISS in retrieval?",               "expected": ["2506.06962v3.pdf"]},
    {"q": "What evaluation metrics are used for image generation?","expected": ["2506.06962v3.pdf"]},
    {"q": "How does patch-level retrieval differ from image-level retrieval?", "expected": ["2506.06962v3.pdf"]},
    {"q": "What is the recipe for chocolate cake?",                "expected": None},
    {"q": "Who won the 2022 FIFA World Cup?",                      "expected": None},
]
print(len(eval_set), "questions")

10 questions


In [18]:
rows = []
for item in eval_set:
    res = search(item["q"], K)
    srcs = [r["source"] for r in res]
    exp = item["expected"]
    hit1 = hit5 = None
    if isinstance(exp, list):
        hit1 = srcs[0] in exp
        hit5 = any(s in exp for s in srcs)
    rows.append({"question": item["q"], "top_score": round(res[0]["score"], 3),
                 "top_source": srcs[0], "hit@1": hit1, "hit@5": hit5})

df_ret = pd.DataFrame(rows)
display(df_ret)

auto = df_ret[df_ret["hit@1"].notna()]
print(f"Hit@1: {auto['hit@1'].astype(bool).mean():.0%}   Hit@{K}: {auto['hit@5'].astype(bool).mean():.0%}   (n={len(auto)})")

ans_scores = [r["top_score"] for r, it in zip(rows, eval_set) if isinstance(it["expected"], list)]
neg_scores = [r["top_score"] for r, it in zip(rows, eval_set) if it["expected"] is None]
print("Lowest top-score of answerable questions:", min(ans_scores))
print("Highest top-score of unanswerable questions:", max(neg_scores))
print("A threshold between these two values can reject off-topic questions without calling Gemini.")

,question,top_score,top_source,hit@1,hit@5
0,What is autoregressive retrieval augmentation?,0.550,2506.06962v3.pdf,True,True
1,How does MusT-RAG handle music question answer...,0.701,2507.23334v2.pdf,True,True
2,What datasets were used to evaluate the AR-RAG...,0.540,2506.06962v3.pdf,True,True
3,What is the main limitation of standard RAG sy...,0.452,2510.25621v1.pdf,True,True
4,How does agentic RAG improve Quran question an...,0.747,2601.07528v2.pdf,True,True
5,What is the role of FAISS in retrieval?,0.478,2510.25621v1.pdf,False,False
6,What evaluation metrics are used for image gen...,0.616,2506.06962v3.pdf,True,True
7,How does patch-level retrieval differ from ima...,0.651,2506.06962v3.pdf,True,True
8,What is the recipe for chocolate cake?,0.201,1912.02145v1.pdf,None,None
9,Who won the 2022 FIFA World Cup?,0.226,2601.07528v2.pdf,None,None


Hit@1: 88%   Hit@5: 88%   (n=8)
Lowest top-score of answerable questions: 0.452
Highest top-score of unanswerable questions: 0.226
A threshold between these two values can reject off-topic questions without calling Gemini.


In [19]:
must_have = {
    "What datasets were used to evaluate the AR-RAG paper?": ["midjourney", "geneval", "dpg"],
    "What is the role of FAISS in retrieval?": ["faiss"],
}

def context_has(q, kws, k):
    text = " ".join(r["text"].lower() for r in search(q, k))
    return all(kw in text for kw in kws)

print("Checking if the new chunks contain the required keywords:")
for q, kws in must_have.items():
    results = {k: context_has(q, kws, k) for k in (5, 8, 10)}
    print(f"{results} | {q}")

Checking if the new chunks contain the required keywords:
{5: True, 8: True, 10: True} | What datasets were used to evaluate the AR-RAG paper?
{5: False, 8: False, 10: False} | What is the role of FAISS in retrieval?


In [5]:
for q in ["What is the main limitation of standard RAG systems?",
          "What is the role of FAISS in retrieval?"]:
    print("=" * 80)
    print(q)
    for i, r in enumerate(search(q, 5), 1):
        print(f"\n[{i}] {r['score']:.3f} | {r['source']}")
        print(r["text"][:300].replace("\n", " "))

from collections import Counter
print("\n" + "=" * 80)
for kw in ["faiss", "vector database", "vector store"]:
    c = Counter(ch["source"] for ch in all_chunks if kw in ch["text"].lower())
    print(f"'{kw}' appears in:", dict(c))

What is the main limitation of standard RAG systems?

[1] 0.517 | 2510.25621v1.pdf
o not have to re- sult in system-level failure. The iterative re- finement cycle of FAIR-RAG acts as a critical compensation mechanism, intelligently adapt- ing its search strategy to overcome initial re- trieval weaknesses and progressively build the complete evidence base required for a faithful a

[2] 0.517 | 2510.25621v1.pdf
tion). 5.4 Baselines and Ablation Studies To isolate the benefits of the FAIR-RAG archi- tecture, we compare the full FARSIQA system against several ablated versions: • FARSIQA (Full System - Optimal Setting): The complete implementa- tion with a maximum of 3 iterations (max_iter=3), which was selec

[3] 0.513 | 2601.07528v2.pdf
Standard RAG Agentic RAG Avg. tokens 502 1,520 Avg. tool calls 0 6 Avg. latency (s) 0.52 3.45 Estimated latency increase 1× 6.63× Table 6: Inference-time efficiency comparison between Standard RAG and Agentic RAG for Qwen3-4B. Agentic RAG Analysis. Agenti

In [6]:
for ch in all_chunks:
    if "faiss" in ch["text"].lower():
        print(ch["source"])
        print(ch["text"])

2506.06962v3.pdf
lion from JourneyDB [36], and 4.6 million from
DataComp [12], while ensuring that any samples included in the testing set are excluded to prevent
data leakage. Each image is encoded into a sequence of patch-level representations and image tokens
using the same image tokenizer employed in the Janus-Pro model. For efficient similarity search, we
implement our retriever using the FAISS library [21].
Training Setup
We adopt Janus-Pro-1B [9] and Show-o [44] as our backbone models and fine-tune
them o


In [26]:

RESULTS_F = f"{DATA}/eval_results_v2.json"

done = {}
if os.path.exists(RESULTS_F):
    for r in json.load(open(RESULTS_F)):
        if r.get("answer"):
            done[r["question"]] = r

eval_results, quota_hit = [], False
for i, item in enumerate(eval_set, 1):
    q = item["q"]
    if q in done:
        eval_results.append(done[q])
        print(f"[{i}/{len(eval_set)}] cached: {q}")
        continue
    if quota_hit:
        continue
    print(f"[{i}/{len(eval_set)}] asking: {q}")
    try:
        answer, res = ask(q)
        eval_results.append({"question": q, "answer": answer,
                             "top_score": res[0]["score"], "top_source": res[0]["source"]})
        print("   ", answer[:120].replace("\n", " "))
    except Exception as e:
        print("   FAILED:", str(e)[:200])
        if "429" in str(e) or "RESOURCE_EXHAUSTED" in str(e):
            quota_hit = True
            print("   Quota reached. Run this cell again later; finished answers are saved.")
    time.sleep(3)

with open(RESULTS_F, "w") as f:
    json.dump(eval_results, f, indent=2)
print(f"\nSaved {len(eval_results)} of {len(eval_set)} results")

[1/10] cached: What is autoregressive retrieval augmentation?
[2/10] cached: How does MusT-RAG handle music question answering?
[3/10] cached: What datasets were used to evaluate the AR-RAG paper?
[4/10] cached: What is the main limitation of standard RAG systems?
[5/10] cached: How does agentic RAG improve Quran question answering?
[6/10] cached: What is the role of FAISS in retrieval?
[7/10] asking: What evaluation metrics are used for image generation?
    Based on the provided context, the evaluation metrics used for image generation (specifically evaluated on the Midjourne
[8/10] cached: How does patch-level retrieval differ from image-level retrieval?
[9/10] cached: What is the recipe for chocolate cake?
[10/10] asking: Who won the 2022 FIFA World Cup?
   transient error, retrying in 5s...
    I don't have enough information.

Saved 10 of 10 results


In [25]:
RESULTS_F = f"{DATA}/eval_results_v2.json"
data = json.load(open(RESULTS_F))
data = [r for r in data if not r["question"].startswith(("What evaluation metrics", "Who won"))]
json.dump(data, open(RESULTS_F, "w"), indent=2)
print("kept", len(data), "real answers")

kept 8 real answers


In [27]:
def refused(a):
    a = a.lower()
    return "don't have enough information" in a or "do not have enough information" in a

res = json.load(open(f"{DATA}/eval_results_v2.json"))
by_q = {r["question"]: r for r in res}
ans_items = [e for e in eval_set if e["expected"]]
off_items = [e for e in eval_set if e["expected"] is None]

missing = [e["q"] for e in eval_set if e["q"] not in by_q]
answered = [e["q"] for e in ans_items if e["q"] in by_q and not refused(by_q[e["q"]]["answer"])]
off_ok = [e["q"] for e in off_items if e["q"] in by_q and refused(by_q[e["q"]]["answer"])]

print("Missing answers:", missing or "none")
print(f"Answerable questions answered: {len(answered)}/{len(ans_items)} ({len(answered)/len(ans_items):.0%})")
print(f"Off-topic questions refused:   {len(off_ok)}/{len(off_items)}")

pd.DataFrame(res).to_csv(f"{DATA}/eval_scored_v2.csv", index=False)

print("\nAnswers to grade:")
for e in ans_items:
    r = by_q.get(e["q"])
    if r and not refused(r["answer"]):
        print("\nQ:", e["q"], "\n", r["answer"], "\n", "-" * 70)

Missing answers: none
Answerable questions answered: 7/8 (88%)
Off-topic questions refused:   2/2

Answers to grade:

Q: What is autoregressive retrieval augmentation? 
 Autoregressive Retrieval Augmentation (AR-RAG) is a paradigm for image generation that enhances the generation process by autoregressively incorporating $k$-nearest neighbor retrievals at the patch level [3]. In this method, image patches function as retrieval queries and keys to retrieve patch values that assist in predicting the next image patches [1, 3], addressing limitations found in conventional image-level retrieval augmentation approaches [4]. 
 ----------------------------------------------------------------------

Q: How does MusT-RAG handle music question answering? 
 MusT-RAG handles music question answering by using a Retrieval-Augmented Generation (RAG) framework to adapt general-purpose large language models (LLMs) to the music domain [1], [3], [4]. Specifically, it operates by:

1. **Retrieving relevant

In [28]:
base_scores = {0: 1, 1: 0.5, 2: 0, 3: 0, 4: 1, 5: 0, 6: 1, 7: 1, 8: 1, 9: 1}   
v2_scores   = {0: 1, 1: 1,   2: 1, 3: 1, 4: 1, 5: 0, 6: 1, 7: 1, 8: 1, 9: 1}   

def summary(scores, label, hit1, answered):
    ans = [scores[i] for i, e in enumerate(eval_set) if e["expected"]]
    return {"setup": label, "Hit@1": hit1, "answered": answered,
            "answer score (answerable only)": round(float(np.mean(ans)), 2),
            "overall score (all 10)": round(float(np.mean(list(scores.values()))), 2),
            "fully correct": f"{sum(s == 1 for s in ans)}/{len(ans)}"}

table = pd.DataFrame([
    summary(base_scores, "baseline (500-char chunks)", "88%", "5/8 (62%)"),
    summary(v2_scores,   "v2 (sentence-aware chunks)", "88%", "7/8 (88%)"),
])
display(table)
table.to_csv(f"{DATA}/comparison_baseline_vs_v2.csv", index=False)
print("Saved ../data/comparison_baseline_vs_v2.csv")

,setup,Hit@1,answered,answer score (answerable only),overall score (all 10),fully correct
0,baseline (500-char chunks),88%,5/8 (62%),0.56,0.65,4/8
1,v2 (sentence-aware chunks),88%,7/8 (88%),0.88,0.90,7/8


Saved ../data/comparison_baseline_vs_v2.csv


In [29]:
from rank_bm25 import BM25Okapi

tok = lambda s: re.findall(r"[a-z0-9\-]+", s.lower())
bm25 = BM25Okapi([tok(c["text"]) for c in all_chunks])

def hybrid_search(query, k=5, pool=30, rrf_k=60):
    q = embedder.encode([query], normalize_embeddings=True)
    sc, ix = index.search(np.array(q, dtype="float32"), pool)
    dense = [int(i) for i in ix[0]]
    sparse = [int(i) for i in np.argsort(bm25.get_scores(tok(query)))[::-1][:pool]]
    fused = {}
    for ranks in (dense, sparse):
        for r, i in enumerate(ranks):
            fused[i] = fused.get(i, 0) + 1 / (rrf_k + r + 1)
    top = sorted(fused, key=fused.get, reverse=True)[:k]
    return [{"score": float(np.dot(q[0], index.reconstruct(i))),
             "text": all_chunks[i]["text"], "source": all_chunks[i]["source"]} for i in top]

for name, fn in [("dense only", search), ("hybrid", hybrid_search)]:
    n = h1 = h5 = 0
    for e in eval_set:
        if not e["expected"]:
            continue
        srcs = [r["source"] for r in fn(e["q"], 5)]
        n += 1
        h1 += srcs[0] in e["expected"]
        h5 += any(s in e["expected"] for s in srcs)
    print(f"{name:11s} Hit@1 {h1}/{n}   Hit@5 {h5}/{n}")

faiss_q = "What is the role of FAISS in retrieval?"
for name, fn in [("dense only", search), ("hybrid", hybrid_search)]:
    found = any("faiss" in r["text"].lower() for r in fn(faiss_q, 5))
    print(f"{name:11s} FAISS passage in top 5: {found}")

# Off-topic questions must still stay below the threshold
for q in ["What is the recipe for chocolate cake?", "Who won the 2022 FIFA World Cup?"]:
    print(round(max(r["score"] for r in hybrid_search(q, 5)), 3), "|", q)

dense only  Hit@1 7/8   Hit@5 7/8
hybrid      Hit@1 7/8   Hit@5 7/8
dense only  FAISS passage in top 5: False
hybrid      FAISS passage in top 5: False
0.201 | What is the recipe for chocolate cake?
0.226 | Who won the 2022 FIFA World Cup?


In [30]:
STOP = set("what is the role of in a an and or to how does do are were was for with on by from this that these those it its as at be used use".split())
tok2 = lambda s: [w for w in tok(s) if w not in STOP]
bm25b = BM25Okapi([tok2(c["text"]) for c in all_chunks])

target = next(i for i, c in enumerate(all_chunks) if "faiss" in c["text"].lower())
qv = embedder.encode([faiss_q], normalize_embeddings=True)
sc, ix = index.search(np.array(qv, dtype="float32"), len(all_chunks))

print("Rank of the FAISS chunk out of", len(all_chunks), "(1 = best):")
print("  dense embeddings:        ", list(ix[0]).index(target) + 1)
print("  BM25 (all words):        ", np.argsort(bm25.get_scores(tok(faiss_q)))[::-1].tolist().index(target) + 1)
print("  BM25 (stopwords removed):", np.argsort(bm25b.get_scores(tok2(faiss_q)))[::-1].tolist().index(target) + 1)

Rank of the FAISS chunk out of 726 (1 = best):
  dense embeddings:         189
  BM25 (all words):         3
  BM25 (stopwords removed): 1


In [31]:
def hybrid_w(query, k=5, pool=30, rrf_k=60, w_sparse=2.0):
    q = embedder.encode([query], normalize_embeddings=True)
    sc, ix = index.search(np.array(q, dtype="float32"), pool)
    dense = [int(i) for i in ix[0]]
    sparse = [int(i) for i in np.argsort(bm25b.get_scores(tok2(query)))[::-1][:pool]]
    fused = {}
    for r, i in enumerate(dense):
        fused[i] = fused.get(i, 0) + 1 / (rrf_k + r + 1)
    for r, i in enumerate(sparse):
        fused[i] = fused.get(i, 0) + w_sparse / (rrf_k + r + 1)
    top = sorted(fused, key=fused.get, reverse=True)[:k]
    return [{"score": float(np.dot(q[0], index.reconstruct(i))),
             "text": all_chunks[i]["text"], "source": all_chunks[i]["source"]} for i in top]

ans_q = [e for e in eval_set if e["expected"]]
print("w_sparse | Hit@1 | Hit@5 | FAISS in top5 | questions whose top-5 passages are unchanged vs dense")
for w in (1, 2, 3, 5):
    h1 = h5 = same = 0
    for e in ans_q:
        res = hybrid_w(e["q"], 5, w_sparse=w)
        srcs = [r["source"] for r in res]
        h1 += srcs[0] in e["expected"]
        h5 += any(s in e["expected"] for s in srcs)
        same += {r["text"] for r in res} == {r["text"] for r in search(e["q"], 5)}
    found = any("faiss" in r["text"].lower() for r in hybrid_w(faiss_q, 5, w_sparse=w))
    print(f"   {w}     | {h1}/{len(ans_q)}  | {h5}/{len(ans_q)}  |    {found}      | {same}/{len(ans_q)}")

for q in ["What is the recipe for chocolate cake?", "Who won the 2022 FIFA World Cup?"]:
    print(round(max(r["score"] for r in hybrid_w(q, 5, w_sparse=3)), 3), "|", q)

w_sparse | Hit@1 | Hit@5 | FAISS in top5 | questions whose top-5 passages are unchanged vs dense
   1     | 7/8  | 7/8  |    False      | 0/8
   2     | 7/8  | 8/8  |    True      | 0/8
   3     | 7/8  | 8/8  |    True      | 0/8
   5     | 7/8  | 8/8  |    True      | 0/8
0.145 | What is the recipe for chocolate cake?
0.205 | Who won the 2022 FIFA World Cup?


In [33]:
HYB_F = f"{DATA}/eval_results_v2_hybrid.json"
search_dense = globals().get("search_dense", search)            # remember the original dense search
search = lambda query, k=5: hybrid_w(query, k, w_sparse=2)      # ask() now uses hybrid search

done = {}
if os.path.exists(HYB_F):
    done = {r["question"]: r for r in json.load(open(HYB_F)) if r.get("answer")}

hyb_results, stop = [], False
for i, item in enumerate(eval_set, 1):
    q = item["q"]
    if q in done:
        hyb_results.append(done[q]); print(f"[{i}/10] cached: {q}"); continue
    if stop:
        continue
    top = max(r["score"] for r in search(q, 5))
    if top < 0.30:
        hyb_results.append({"question": q, "answer": "I don't have enough information. (score gate)",
                            "top_score": top, "top_source": "-"})
        print(f"[{i}/10] gated, no Gemini call: {q}"); continue
    for attempt in range(3):
        print(f"[{i}/10] asking: {q}")
        try:
            answer, res = ask(q, max_retries=2)
            hyb_results.append({"question": q, "answer": answer, "top_score": top, "top_source": res[0]["source"]})
            print("   ", answer[:120].replace("\n", " "))
            break
        except Exception as e:
            msg = str(e)
            if "PerDay" in msg:
                print("   DAILY quota reached. Stop and run this cell again after the reset.")
                stop = True
                break
            if "429" in msg or "RESOURCE_EXHAUSTED" in msg:
                print("   per-minute limit, waiting 65 s...")
                time.sleep(65)
                continue
            print("   FAILED:", msg[:200])
            break
    time.sleep(13)    # the free tier allows only 5 requests per minute

search = search_dense                                            # restore dense search
with open(HYB_F, "w") as f:
    json.dump(hyb_results, f, indent=2)
print(f"\nSaved {len(hyb_results)} of {len(eval_set)} results")

[1/10] asking: What is autoregressive retrieval augmentation?
   FAILED: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.
   Quota reached. Run this cell again later; finished answers are saved.

Saved 0 of 10 results


In [34]:
try:
    r = llm.models.generate_content(model=GEN_MODEL, contents="Say hi")
    print("WORKS:", r.text)
except Exception as e:
    print(str(e)[:1800])

429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.8-flash\nPlease retry in 4h23m51.548469586s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-3.8-flash'}, 'quotaV